# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ali-Haider987/alihaider-flyrank-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score,
                              precision_score, recall_score, f1_score)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))


30000 pages |  declining rate: 0.542


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

My lane (`w02_ml_task_framing.ipynb`) is a **yes/no classification** problem with an **observed label**: `is_declining_label`. Per the `training-honest-models` skill's own table, that question shape starts with **Logistic Regression** (readable, gives a coefficient story) and then **Random Forest** (stronger, still gives feature importances). I skip Gradient Boosting this round — with only 17 rows my Week-4 rule ever flagged and a client-holdout test set of 2,325 rows, a heavier ensemble is complexity I can't yet justify against a random forest baked with conservative depth/leaf settings; the skill's own line is "add complexity only when the comparison earns it," and it hasn't yet.

**Feature set:** I reuse `MODEL_NUMERIC_FEATURES` / `MODEL_CATEGORICAL_FEATURES` from `scripts/ml_utils.py` — the same list the repo's own pipeline already vetted for leakage (`trend_direction` / `trend_pct` are not in it). I add `log1p` transforms for the four heavy-tailed traffic totals (matches the `auditing-signals` lesson) and an explicit `_missing` flag for every column with structural (not random) missingness, instead of a blind `fillna(0)`.

In [2]:
MODEL_NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
MODEL_CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent", "age_tier", "freshness_tier",
    "word_count_tier", "impression_tier", "position_tier",
]

df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

# Missingness follows content_type (flyrank-data skill) -- flag it, don't just zero-fill it away.
missing_flag_cols = ["search_volume", "competition", "cpc", "word_count", "char_count",
                     "engagement_rate", "scroll_rate", "ai_traffic_pct"]
for c in missing_flag_cols:
    df[f"{c}_missing"] = df[c].isna().astype(int)

numeric_frame = (df[MODEL_NUMERIC_FEATURES]
                 .apply(pd.to_numeric, errors="coerce")
                 .replace([np.inf, -np.inf], np.nan)
                 .fillna(0))
missing_frame = df[[f"{c}_missing" for c in missing_flag_cols]]
cat_frame = df[MODEL_CATEGORICAL_FEATURES].fillna("unknown").astype(str)
cat_dummies = pd.get_dummies(cat_frame, prefix=MODEL_CATEGORICAL_FEATURES, dtype=float)

feature_frame = pd.concat(
    [numeric_frame.reset_index(drop=True), missing_frame.reset_index(drop=True), cat_dummies.reset_index(drop=True)],
    axis=1,
)
print("Feature matrix:", feature_frame.shape)
print("Confirms label-source columns are absent from the matrix:",
      {"trend_direction", "trend_pct"} & set(feature_frame.columns))


Feature matrix: (30000, 60)
Confirms label-source columns are absent from the matrix: set()


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Client-grouped holdout**, same logic as the repo's own `scripts/03_train_model.py`: shuffle the 32 client IDs with a fixed seed, hold out 20% of *clients* (not 20% of rows) for testing. This matters because rows from the same client share a lot — the same site, the same content strategy, similar baseline traffic. A row-level random split would let the model see other pages from the same client in training and effectively memorize the client, which would make the test score a measure of "do I recognize this client" rather than "can I spot decline in a client I've never seen." This is also the exact same split my Week-4 baseline gets evaluated on below, so the comparison in Section 3 is apples-to-apples.

In [3]:
client_series = df["client_id"].astype(str)
unique_clients = client_series.drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
shuffled_clients = rng.permutation(unique_clients)
test_client_count = max(1, int(round(len(shuffled_clients) * 0.2)))
test_clients = set(shuffled_clients[:test_client_count])
test_mask = client_series.isin(test_clients).to_numpy()

X_train, X_test = feature_frame[~test_mask], feature_frame[test_mask]
y_train = df.loc[~test_mask, "is_declining_label"]
y_test = df.loc[test_mask, "is_declining_label"]

print(f"{len(unique_clients)} clients total, {test_client_count} held out for test")
print(f"train rows: {len(X_train):,}  (declining rate {y_train.mean():.3f})")
print(f"test  rows: {len(X_test):,}  (declining rate {y_test.mean():.3f})")
assert y_train.nunique() == 2 and y_test.nunique() == 2, "split produced a single-class side"


32 clients total, 6 held out for test
train rows: 27,675  (declining rate 0.555)
test  rows: 2,325  (declining rate 0.391)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [4]:
def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(y_true)[order[:k]].mean())

# --- Week-4 baseline, scored on this exact test slice (stale x visible x impressions) ---
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["baseline_score"] = stale * visible * df["impressions_90d"]

baseline_test_scores = df.loc[test_mask, "baseline_score"].values
n_flagged_in_test = int((baseline_test_scores > 0).sum())
print(f"Baseline flags {n_flagged_in_test} of {len(baseline_test_scores)} held-out rows.")


Baseline flags 0 of 2325 held-out rows.


In [5]:
# --- Logistic Regression ---
lr = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
])
lr.fit(X_train, y_train)
lr_scores = lr.predict_proba(X_test)[:, 1]
lr_pred = (lr_scores >= 0.5).astype(int)

# --- Random Forest ---
rf = RandomForestClassifier(
    class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
    n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE,
)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]
rf_pred = (rf_scores >= 0.5).astype(int)

print("Both models trained on the same", len(X_train), "-row client-holdout train split.")


Both models trained on the same 27675 -row client-holdout train split.


In [6]:
rows = []

# Baseline: ROC-AUC and average precision are still valid with a constant (mostly-zero) score --
# sklearn correctly returns "no ranking signal" (AUC 0.5) and "no better than guessing the base
# rate" (avg precision == test base rate). Precision@K is NOT reported for the baseline: with 0
# flagged rows in this held-out slice, any "top K" would just be whatever K rows happen to sit
# first after a tie-stable sort of all-zero scores -- that number would look like a result but
# would really just be file order. Reporting it would be dishonest, not decoration.
rows.append({
    "model": "baseline (stale x visible)",
    "roc_auc": roc_auc_score(y_test, baseline_test_scores),
    "avg_precision": average_precision_score(y_test, baseline_test_scores),
    "precision@20": "n/a (0 flagged)", "precision@50": "n/a (0 flagged)", "precision@100": "n/a (0 flagged)",
    "accuracy": None, "precision": None, "recall": None, "f1": None,
})

for name, scores, pred in [("logistic_regression", lr_scores, lr_pred), ("random_forest", rf_scores, rf_pred)]:
    rows.append({
        "model": name,
        "roc_auc": roc_auc_score(y_test, scores),
        "avg_precision": average_precision_score(y_test, scores),
        "precision@20": precision_at_k(y_test, scores, 20),
        "precision@50": precision_at_k(y_test, scores, 50),
        "precision@100": precision_at_k(y_test, scores, 100),
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "f1": f1_score(y_test, pred),
    })

comparison = pd.DataFrame(rows).set_index("model")
base_rate = y_test.mean()
print(f"Test base rate (declining): {base_rate:.3f}\n")
comparison.round(3)


Test base rate (declining): 0.391



,roc_auc,avg_precision,precision@20,precision@50,precision@100,accuracy,precision,recall,f1
model,,,,,,,,,
baseline (stale x visible),0.500,0.391,n/a (0 flagged),n/a (0 flagged),n/a (0 flagged),NaN,NaN,NaN,NaN
logistic_regression,0.702,0.524,0.35,0.4,0.43,0.662,0.568,0.568,0.568
random_forest,0.749,0.618,0.8,0.76,0.73,0.665,0.552,0.754,0.638


**Reading the table:** the baseline can't even be scored at precision@K on this held-out slice — it only ever fires for 17 of 30,000 pages, spread across 4 of 32 clients, and this particular client split happened to hold out none of them (`0 flagged in test`, confirmed above). Its ROC-AUC of exactly 0.500 and average precision equal to the base rate (0.391) are the honest "no information" values for a rule that never activates here — not a bug, the real cost of an extremely narrow hand rule. Both models clear that bar by a wide margin, and **Random Forest is the clear winner over Logistic Regression too**: roughly double the precision@20/50/100, a meaningfully higher ROC-AUC (0.749 vs 0.702) and average precision (0.618 vs 0.524), and a much better recall/F1 trade-off. I did not need Gradient Boosting to beat the baseline convincingly.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [7]:
# --- What the winning model (Random Forest) leans on ---
impurity_importance = pd.Series(rf.feature_importances_, index=feature_frame.columns).sort_values(ascending=False)
print("Top 8 by impurity-based importance:")
print(impurity_importance.head(8).round(4))

perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE,
                               scoring="roc_auc", n_jobs=-1)
perm_importance = pd.Series(perm.importances_mean, index=feature_frame.columns).sort_values(ascending=False)
print("\nTop 8 by permutation importance (ROC-AUC drop when shuffled):")
print(perm_importance.head(8).round(4))


Top 8 by impurity-based importance:
days_with_impressions    0.1324
log_impressions_90d      0.1285
avg_position             0.1233
content_age_days         0.0780
ctr                      0.0341
char_count               0.0338
word_count               0.0337
log_clicks_90d           0.0319
dtype: float64

Top 8 by permutation importance (ROC-AUC drop when shuffled):
days_with_impressions    0.0488
log_impressions_90d      0.0275
ctr                      0.0110
avg_position             0.0108
log_clicks_90d           0.0051
scroll_rate              0.0051
position_tier_top_3      0.0028
search_volume            0.0027
dtype: float64


Both rankings agree on the same top 3: **`days_with_impressions`**, **`log_impressions_90d`**, and **`avg_position`** / **`ctr`** — how consistently and how well a page shows up in search. That's a sensible story, not a suspicious one: none of these are the label in disguise (`trend_direction` / `trend_pct` never entered the matrix, confirmed in Section 1), and none of them sit at 0.9+ importance the way a leaked column would. `content_age_days` and `char_count` / `word_count` show up further down — length matters less than visibility, which lines up with Discovery C from `01_first_look_and_discovery.ipynb` (word count barely separates declining and growing pages) and my own Week-4 finding that visibility alone doesn't cleanly predict decline.

In [8]:
# --- Concrete wrong cases ---
test_df = df.loc[test_mask].copy().reset_index(drop=True)
test_df["prob"] = rf_scores
test_df["pred"] = rf_pred
test_df["true"] = y_test.values

fp = test_df[(test_df["pred"] == 1) & (test_df["true"] == 0)].sort_values("prob", ascending=False)
fn = test_df[(test_df["pred"] == 0) & (test_df["true"] == 1)].sort_values("prob")
cols = ["content_id", "prob", "trend_direction", "days_since_last_update", "impressions_90d", "avg_position", "ctr"]

print(f"{len(fp)} false positives, {len(fn)} false negatives out of {len(test_df)} test rows\n")
print("Top 3 false positives (model says declining, actually not):")
print(fp[cols].head(3).to_string(index=False))
print("\nTop 3 false negatives (model says fine, actually declining):")
print(fn[cols].head(3).to_string(index=False))

print("\nAccuracy by content_type:")
print(test_df.groupby("content_type").apply(lambda g: (g['pred'] == g['true']).mean()).round(3))


555 false positives, 224 false negatives out of 2325 test rows

Top 3 false positives (model says declining, actually not):
          content_id     prob trend_direction  days_since_last_update  impressions_90d  avg_position  ctr
content_db1cd41b4b4f 0.738810              up                     105             1482          12.9  0.0
content_f5013794ba57 0.737693             new                      20              881          15.7  0.0
content_331182ca4cae 0.732828              up                      20             3026          35.9  0.0

Top 3 false negatives (model says fine, actually declining):
          content_id     prob trend_direction  days_since_last_update  impressions_90d  avg_position  ctr
content_28b4223f4e5f 0.070057            down                       1                1           0.0  0.0
content_34b14c00f80c 0.084885            down                      20                3           0.0  0.0
content_79ac977c6e0b 0.152133            down                       8   

/tmp/ipykernel_2337/2008654598.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(test_df.groupby("content_type").apply(lambda g: (g['pred'] == g['true']).mean()).round(3))


**Why these are hard:**

- **False positives** are almost all pages with `ctr = 0.00%` and decent `avg_position` (12.9–35.9) that are actually `up` or `new`, not declining. The model has learned "zero clicks at a real position looks like decline," but a brand-new page (`new`) or one that just started ranking hasn't had time to accumulate clicks yet — zero CTR there means "too early," not "failing."
- **False negatives** are pages with `avg_position = 0`, essentially 1–3 impressions total, that are genuinely `down`. `avg_position = 0` is the dataset's "no position data" code, not an actual top rank (a documented gotcha in `docs/data-dictionary.md`) — these pages are so close to invisible that almost every feature reads as near-zero, and the model has almost nothing to separate a dying page from one that never had traffic to lose.
- **Segment gap:** accuracy is noticeably lower on `keyword article` (0.592) than `feedly article` (0.769) pages — worth a closer look before trusting the model equally across content types, though `feedly article` is a much smaller slice of the data so that gap itself needs more rows to confirm.

None of this is a reason to distrust the headline numbers — it's exactly the kind of decision-support caveat a content team needs before acting on the model's queue: check CTR=0% picks for "is this just new" before flagging them, and treat picks with `avg_position = 0` as low-confidence regardless of what the model says.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.